In [6]:
import pandas as pd

df = pd.read_csv("RetailCleaned.csv")

df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

C:\Users\sriya\AppData\Local\Temp\ipykernel_29224\4254881002.py:3: DtypeWarning: Columns (0: 489434) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("RetailCleaned.csv")


(1033035, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033035 entries, 0 to 1033034
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033035 non-null  object        
 1   StockCode    1033035 non-null  str           
 2   Description  1028760 non-null  str           
 3   Quantity     1033035 non-null  int64         
 4   InvoiceDate  1033035 non-null  datetime64[ns]
 5   Price        1033035 non-null  float64       
 6   Customer_ID  797884 non-null   float64       
 7   Country      1033035 non-null  str           
 8   SourceSheet  1033035 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 70.9+ MB


In [15]:
#Filter and Calculate Recency, Frequency, Monetary 
rfm_data = df[df['Customer_ID'].notna() & (df['Quantity']>0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
  Recency = ("InvoiceDate", lambda x : (snapshot_date - x.max()).days),
  Frequency = ('Invoice', 'nunique'),
  Monetary = ('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()
print('rfm data', rfm.head(5))


rfm data    Customer_ID  Recency  Frequency  Monetary
0      12346.0      325         12  77556.46
1      12347.0        2          8   4921.53
2      12348.0       75          5   2019.40
3      12349.0       18          4   4428.69
4      12350.0      310          1    334.40


In [24]:
#Score customers and assign business segments
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels = [5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method = 'first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels= [1,2,3,4,5]).astype(int)

def segment_customer(row):
  if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
    return 'Champions'
  elif row['R_score'] >= 3 and row['F_score'] >= 3:
    return 'Loyal'
  elif row['R_score'] >= 2 and row['F_score'] >= 3:
    return 'At risk'
  elif row['R_score'] >= 4 and row['F_score'] >= 2:
    return 'New'
  elif row['R_score'] >= 3 and row['F_score'] >= 2:
    return 'Needs attention'
  else:
    return 'lost'

rfm['Segment'] = rfm.apply(segment_customer, axis =1)
rfm['IsChurned'] = rfm['Recency']>90
rfm

,Customer_ID,Recency,Frequency,Monetary,R_score,F_score,M-score,M_score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,5,At risk,True
1,12347.0,2,8,4921.53,5,4,5,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,4,Loyal,False
3,12349.0,18,4,4428.69,5,3,5,5,Loyal,False
4,12350.0,310,1,334.40,2,1,2,2,lost,True
...,...,...,...,...,...,...,...,...,...,...
5876,18283.0,3,22,2664.90,5,5,4,4,Champions,False
5877,18284.0,431,1,461.68,1,2,2,2,lost,True
5878,18285.0,660,1,427.00,1,2,2,2,lost,True
5879,18286.0,476,2,1296.43,1,3,4,4,lost,True


In [ ]:
#MERGE REM RESULTS BACK ONTO TRANSACTIONS AND EXPORT
df =df[(df['Customer_ID'].notna())&(df['Quantity']>0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
  rfm[['Customer_ID', 'Recency',	'Frequency',	'Monetary',	'R_score',	'F_score',	'M-score', 'Segment', 'IsChurned' ]],
  on = 'Customer_ID', how = 'left'
)
df_final.to_excel('EcommerceDashboardData.xlsx', index = False)